# 03 — QAOA Comparison (Optional Advanced Task)

**WISER Quantum+AI 2026 — Moderna Challenge**

This notebook addresses the optional advanced task: *"Compare multiple quantum
encodings of the same folding problem and discuss tradeoffs in qubit count and
constraint enforcement."* We compare three solvers on the **same QUBO / Hamiltonian**
for the same sequences used in notebook 02:

1. **CVaR-VQE** (notebook 02's primary method) — hardware-efficient ansatz, CVaR
   objective (alpha=0.1).
2. **Plain QAOA** — problem-derived ansatz (cost + mixer unitaries), standard full
   expectation-value objective.
3. **CVaR-QAOA** — QAOA's ansatz with the same CVaR objective as (1), isolating
   whether the ansatz choice or the objective choice matters more.

Since all three solve the identical Hamiltonian, qubit count is identical across
methods for a given sequence — what differs is circuit structure, parameter count,
and how each converges.

In [1]:
import sys, time
sys.path.insert(0, "..")

import RNA
from classical.evaluate_energy import evaluate_structure_energy, calculate_energy_gap
from quantum.qubo import build_qubo_matrix
from quantum.vqe_solver import solve_cvar_vqe, build_ansatz
from quantum.qaoa_solver import solve_qaoa

## 1. Run all three solvers on the toy sequence

In [2]:
def run_all(sequence, seed=42):
    qubo, pairs = build_qubo_matrix(sequence)
    ref_struct, ref_mfe = RNA.fold(sequence)

    runs = {}

    t0 = time.time()
    runs["CVaR-VQE"] = solve_cvar_vqe(qubo, pairs, len(sequence), alpha=0.1, seed=seed)
    runs["CVaR-VQE"]["runtime"] = time.time() - t0

    t0 = time.time()
    runs["QAOA"] = solve_qaoa(qubo, pairs, len(sequence), aggregation=None, seed=seed)
    runs["QAOA"]["runtime"] = time.time() - t0

    t0 = time.time()
    runs["CVaR-QAOA"] = solve_qaoa(qubo, pairs, len(sequence), aggregation=0.1, seed=seed)
    runs["CVaR-QAOA"]["runtime"] = time.time() - t0

    print(f"=== {sequence} ({len(pairs)} qubits) ===")
    print(f"Reference MFE: {ref_struct}  ({ref_mfe:.2f} kcal/mol)\n")
    for name, r in runs.items():
        pred_energy = evaluate_structure_energy(sequence, r["structure"])
        gap = calculate_energy_gap(pred_energy, ref_mfe)
        match = r["structure"] == ref_struct
        print(f"{name:10s} | structure: {r['structure']:>12s} | "
              f"energy: {pred_energy:6.2f} | match: {str(match):5s} | "
              f"gap: {gap['absolute_gap_kcal']:5.2f} kcal/mol | "
              f"runtime: {r['runtime']:.2f}s")
    print()
    return runs

toy_runs = run_all("GCGCAUACGC")
short_runs = run_all("AUGCAUGC")

=== GCGCAUACGC (7 qubits) ===
Reference MFE: (((....)))  (-1.30 kcal/mol)

CVaR-VQE   | structure:   (((....))) | energy:  -1.30 | match: True  | gap:  0.00 kcal/mol | runtime: 0.82s
QAOA       | structure:   (((....))) | energy:  -1.30 | match: True  | gap:  0.00 kcal/mol | runtime: 0.51s
CVaR-QAOA  | structure:   (((....))) | energy:  -1.30 | match: True  | gap:  0.00 kcal/mol | runtime: 0.43s



=== AUGCAUGC (3 qubits) ===
Reference MFE: ........  (0.00 kcal/mol)

CVaR-VQE   | structure:     (....).. | energy:   5.30 | match: False | gap:  5.30 kcal/mol | runtime: 0.18s
QAOA       | structure:     .(....). | energy:   5.20 | match: False | gap:  5.20 kcal/mol | runtime: 0.33s
CVaR-QAOA  | structure:     ..(....) | energy:   4.50 | match: False | gap:  4.50 kcal/mol | runtime: 0.16s



## 2. Circuit structure comparison

Qubit count is identical (fixed by the Hamiltonian), but ansatz structure differs:

In [3]:
qubo, pairs = build_qubo_matrix("GCGCAUACGC")
n = len(pairs)

vqe_ansatz = build_ansatz(n, reps=2)
print(f"CVaR-VQE ansatz (hardware-efficient, n_local): "
      f"{vqe_ansatz.num_parameters} parameters, depth {vqe_ansatz.depth()}")

qaoa_circuit = toy_runs["QAOA"]["raw_result"].optimal_circuit
print(f"QAOA ansatz (problem-derived, QAOAAnsatz):     "
      f"{qaoa_circuit.num_parameters} parameters, depth {qaoa_circuit.decompose(reps=3).depth()}")

CVaR-VQE ansatz (hardware-efficient, n_local): 21 parameters, depth 11
QAOA ansatz (problem-derived, QAOAAnsatz):     4 parameters, depth 63


## 3. Discussion: tradeoffs in qubit count and constraint enforcement

**Qubit count:** identical across all three methods, since it's fixed by the QUBO
formulation (one qubit per candidate base pair), not by the solver. Changing solver
does not change the resource requirement in this formulation, only the search
strategy over the same space.

**Constraint enforcement:** all three methods enforce the "no shared base" and
"no crossing pairs" constraints the same way, since it happens once, in
`quantum.qubo.build_qubo_matrix`, as **soft penalty terms** baked into the QUBO/
Hamiltonian itself (rather than as hard constraints on the quantum circuit). This
means an invalid configuration is possible in principle but is pushed to high
energy, and it applies identically to CVaR-VQE, QAOA, and CVaR-QAOA, because they
all optimize the exact same Hamiltonian. A different encoding strategy, such as one
that hard-codes exclusivity into the ansatz structure itself, could enforce the
constraint natively instead, at the cost of a more specialized (and more complex)
circuit; that is left as future work.

**Ansatz structure:** CVaR-VQE's ansatz is hardware-efficient and problem-agnostic
(RY + CZ layers, same shape regardless of the specific Hamiltonian). QAOA's ansatz
is problem-derived (alternates a cost unitary built directly from the Hamiltonian
with a mixer unitary), which in principle biases the search toward the problem
structure but produces a deeper circuit per repetition for a QUBO with this many
pairwise interaction terms.

**Objective function:** comparing QAOA against CVaR-QAOA (same ansatz, different
objective) isolates the effect of CVaR: both use the identical problem-derived
circuit, so any difference in convergence speed or solution quality is attributable
to the aggregation strategy, not the ansatz.

## 4. On pseudoknots (optional task, cross-reference)

As already noted in notebook 01, this project explicitly excludes pseudoknots
(crossing base pairs) via the non-crossing penalty term in the QUBO. All three
solvers here inherit that exclusion equally, since it's encoded once at the QUBO
level, not per-solver. A pseudoknot-aware formulation would need either additional
penalty terms for crossing interactions or a fundamentally different variable
encoding, and is left as future work rather than attempted here, given the time
constraints of this challenge.